<a href="https://colab.research.google.com/github/Marcelo-Silvestre/MVP_Machine_Learning_Analytics_PUCRIO_40530010056_20260_01/blob/main/Notebooks/MVP_Machine_Learning_Previsao_Ocorrencias_Criminais_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://www.gov.br/mj/pt-br/assuntos/sua-seguranca/seguranca-publica/estatistica/dados-nacionais-1/base-de-dados-e-notas-metodologicas-dos-gestores-estaduais-sinesp-vde-2022-e-2023

In [1]:
#importação das bibliotecas necessárias ao desenvolvimento do MVP
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
#importação das bases de dados

url_2020 = "https://raw.githubusercontent.com/Marcelo-Silvestre/MVP_Machine_Learning_Analytics_PUCRIO_40530010056_20260_01/main/arquivos/BancoVDE_2020.csv"
url_2021 = "https://raw.githubusercontent.com/Marcelo-Silvestre/MVP_Machine_Learning_Analytics_PUCRIO_40530010056_20260_01/main/arquivos/BancoVDE_2021.csv"
url_2022 = "https://raw.githubusercontent.com/Marcelo-Silvestre/MVP_Machine_Learning_Analytics_PUCRIO_40530010056_20260_01/main/arquivos/BancoVDE_2022.csv"
url_2023 = "https://raw.githubusercontent.com/Marcelo-Silvestre/MVP_Machine_Learning_Analytics_PUCRIO_40530010056_20260_01/main/arquivos/BancoVDE_2023.csv"
url_2024 = "https://raw.githubusercontent.com/Marcelo-Silvestre/MVP_Machine_Learning_Analytics_PUCRIO_40530010056_20260_01/main/arquivos/BancoVDE_2024.csv"
url_2025 = "https://raw.githubusercontent.com/Marcelo-Silvestre/MVP_Machine_Learning_Analytics_PUCRIO_40530010056_20260_01/main/arquivos/BancoVDE_2025.csv"

df_2020 = pd.read_csv(url_2020,sep=";",low_memory=False)
df_2021 = pd.read_csv(url_2021,sep=";",low_memory=False)
df_2022 = pd.read_csv(url_2022,sep=";",low_memory=False)
df_2023 = pd.read_csv(url_2023,sep=";",low_memory=False)
df_2024 = pd.read_csv(url_2024,sep=";",low_memory=False)
df_2025 = pd.read_csv(url_2025,sep=";",low_memory=False)

In [3]:
#Base de dados com todo os dados para o MVP
df_vitimas = pd.concat([df_2020,df_2021,df_2022,df_2023,df_2024,df_2025],ignore_index=True)

In [4]:
df_vitimas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4654995 entries, 0 to 4654994
Data columns (total 14 columns):
 #   Column           Dtype  
---  ------           -----  
 0   uf               object 
 1   municipio        object 
 2   evento           object 
 3   data_referencia  object 
 4   agente           object 
 5   arma             object 
 6   faixa_etaria     object 
 7   feminino         float64
 8   masculino        float64
 9   nao_informado    float64
 10  total_vitima     float64
 11  total            float64
 12  total_peso       float64
 13  abrangencia      object 
dtypes: float64(6), object(8)
memory usage: 497.2+ MB


In [5]:
#Alterando o tipo de dado da coluna 'data_referencia' para o tipo datetime
df_vitimas['data_referencia'] = pd.to_datetime(df_vitimas['data_referencia'],dayfirst=True,errors='coerce')

In [6]:
#Restringindo o dataframe apenas com valores maiores que zero
df_vitimas = df_vitimas.loc[df_vitimas['total_vitima']>0,['uf','evento','data_referencia','feminino','masculino','nao_informado','total_vitima']].reset_index()

In [7]:
#Criação da coluna 'data_ano' e 'data_mes'
idx = df_vitimas.columns.get_loc('data_referencia')
df_vitimas.insert(idx + 1,'data_mes',df_vitimas['data_referencia'].dt.month)
df_vitimas.insert(idx+2,'data_ano',df_vitimas['data_referencia'].dt.year)

In [11]:
#Dataframe formatado para uso
df_vitimas = (
    df_vitimas
    .groupby(
        ['uf','evento','data_referencia','data_mes','data_ano'],
        as_index=True
    )
    .agg({
        'feminino':'sum',
        'masculino':'sum',
        'nao_informado':'sum',
        'total_vitima':'sum'
    })
).reset_index().sort_values(['evento','data_ano'])
df_vitimas

,uf,evento,data_referencia,data_mes,data_ano,feminino,masculino,nao_informado,total_vitima
0,AC,Estupro,2020-01-01,1,2020,12.0,1.0,1.0,14.0
1,AC,Estupro,2020-02-01,2,2020,12.0,0.0,0.0,12.0
2,AC,Estupro,2020-03-01,3,2020,5.0,0.0,0.0,5.0
3,AC,Estupro,2020-04-01,4,2020,6.0,0.0,0.0,6.0
4,AC,Estupro,2020-05-01,5,2020,2.0,1.0,0.0,3.0
...,...,...,...,...,...,...,...,...,...
27699,TO,Tentativa de homicídio,2025-08-01,8,2025,1.0,33.0,2.0,36.0
27700,TO,Tentativa de homicídio,2025-09-01,9,2025,6.0,34.0,2.0,42.0
27701,TO,Tentativa de homicídio,2025-10-01,10,2025,3.0,41.0,1.0,45.0
27702,TO,Tentativa de homicídio,2025-11-01,11,2025,0.0,31.0,0.0,31.0


In [12]:
#Base de dados para a previsão do índice do tipo criminal no ano de 2026 em nível nacional
df_vitimas_BR = df_vitimas.groupby(['evento','data_mes','data_ano'])['total_vitima'].sum().reset_index().sort_values(['evento','data_ano'])
df_vitimas_BR

,evento,data_mes,data_ano,total_vitima
0,Estupro,1,2020,2217.0
6,Estupro,2,2020,2052.0
12,Estupro,3,2020,1766.0
18,Estupro,4,2020,1423.0
24,Estupro,5,2020,1461.0
...,...,...,...,...
1199,Tentativa de homicídio,8,2025,2967.0
1205,Tentativa de homicídio,9,2025,2754.0
1211,Tentativa de homicídio,10,2025,2815.0
1217,Tentativa de homicídio,11,2025,2820.0


In [13]:
#Base de dados para a previsão do índice do tipo criminalem 2026 em cada UF
df_vitimas_UF = df_vitimas.groupby(['uf','evento','data_mes','data_ano'])['total_vitima'].sum().reset_index().sort_values(['evento','data_ano'])
df_vitimas_UF

,uf,evento,data_mes,data_ano,total_vitima
0,AC,Estupro,1,2020,14.0
6,AC,Estupro,2,2020,12.0
12,AC,Estupro,3,2020,5.0
18,AC,Estupro,4,2020,6.0
24,AC,Estupro,5,2020,3.0
...,...,...,...,...,...
27679,TO,Tentativa de homicídio,8,2025,36.0
27685,TO,Tentativa de homicídio,9,2025,42.0
27691,TO,Tentativa de homicídio,10,2025,45.0
27697,TO,Tentativa de homicídio,11,2025,31.0
